# Assumption Ops: reviewed updates, replay, and atomic plan commits

Run **Runtime → Run all** in Colab. CPU only; no API keys. This notebook exercises the workflow across source updates and a database restart.

All supplier confirmations here are **synthetic examples explicitly submitted by the caller**. The batch API is a trusted local review boundary, not an authenticated approval service. No ERP, supplier, customer, or payment system is called.

In [ ]:
from collections import defaultdict
import importlib.util
import subprocess
import sys

needs_install = importlib.util.find_spec("assumption_ops") is None
if not needs_install:
    import assumption_ops as installed_ops
    needs_install = not all(hasattr(installed_ops, name) for name in ("ScenarioComparison", "EvidenceUpdate"))
if needs_install:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
        "git+https://github.com/crypticsymmetry/improved-giggle.git",
    ])
    # Refresh only this framework if an older version was imported in this runtime.
    for module_name in list(sys.modules):
        if module_name == "assumption_ops" or module_name.startswith("assumption_ops."):
            del sys.modules[module_name]

from assumption_ops import (
    ATMS, OperationsPipeline, Supply, Order, Policy, Scenario,
    StaleDecisionError, evaluate_plan, load_csv_scenario, write_comparison_csv,
)
from assumption_ops.evidence import EvidenceConflict
from pprint import pprint

from assumption_ops import EvidenceUpdate, StaleSnapshotError, IdempotencyConflict


## 1. Create a persistent local scenario

The database lives in a temporary directory for this demonstration. An existing v0.2 database also gains the new receipt table when opened; existing observations and decisions are preserved.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

workspace = TemporaryDirectory()
db_path = str(Path(workspace.name) / "workflow.sqlite")
ops = OperationsPipeline(db_path)
ops.seed(
    [Supply("stock", "A", 3, 0, 1), Supply("shipment", "A", 7, 2, 1), Supply("substitute", "B", 4, 0, 3)],
    [Order("urgent", "A", 5, 3, 3), Order("standard", "A", 5, 4)],
    Policy(substitutions={"A": ("B",)}),
)
original = ops.plan()
ops.approve_plan(original.plan_id)
delay_snapshot = ops.revisions()
print("Original plan:", original.plan_id)
pprint(delay_snapshot)


## 2. Apply an explicitly reviewed supplier update

`apply_updates` atomically proposes, accepts, and selects the caller's confirmed values as the winners for the named fields. Use ordinary `propose` and `accept` for unreviewed inbox observations instead.

A new external event ID must match both evidence and operations revisions. The event ID and its ordered, source-attributed payload identify one durable delivery. A duplicate returns the original receipt; it never reapplies its older facts.

In [ ]:
delay_updates = [EvidenceUpdate(
    "supply:shipment", "available_day", 8, "demo: reviewed supplier delay confirmation"
)]
delay_receipt = ops.apply_updates("supplier-delay-001", delay_updates,
    expected_evidence_revision=delay_snapshot["evidence_revision"],
    expected_operations_revision=delay_snapshot["operations_revision"])
pprint(delay_receipt.to_dict())
assert not delay_receipt.replayed
assert ops.store.current("supply:shipment", "available_day").value == 8
before_stale_commit = ops.events()
try:
    ops.commit_plan(original.plan_id)
except StaleDecisionError:
    print("Earlier approved plan rejected after source change.")
else:
    raise AssertionError("Old approved allocations must not commit")
assert ops.events() == before_stale_commit


## 3. Prove that a bad batch rolls back completely

A valid first update followed by an invalid field must leave no partial source change, audit event, or receipt.

In [ ]:
before_failed_batch = ops.revisions()
before_evidence_events, before_operations_events = ops.store.events(), ops.events()
try:
    ops.apply_updates("invalid-batch", [
        EvidenceUpdate("supply:shipment", "available_day", 3, "demo: tentative reviewed date"),
        EvidenceUpdate("order:missing", "quantity", 5, "demo: invalid entity"),
    ], expected_evidence_revision=before_failed_batch["evidence_revision"],
       expected_operations_revision=before_failed_batch["operations_revision"])
except ValueError as error:
    print("Expected batch rejection:", error)
else:
    raise AssertionError("An invalid later field must reject the entire batch")
assert ops.revisions() == before_failed_batch
assert ops.store.events() == before_evidence_events
assert ops.events() == before_operations_events
assert ops.store.current("supply:shipment", "available_day").value == 8


## 4. Detect a stale intervention snapshot

First compare an expedite hypothesis. Another plan creation changes the operations revision while the comparison is being reviewed. Applying a newly confirmed event against the old comparison token must fail. Refreshing the comparison supplies a new token; this still does not itself confirm or execute the hypothesis.

In [ ]:
candidate = Scenario("expedite", {"supply:shipment": {"available_day": 3}}, action_cost=40)
comparison = ops.compare_scenarios([candidate], previous=original.plan)
confirmed_updates = [
    EvidenceUpdate("supply:shipment", "available_day", 3, "demo: reviewed expedited arrival confirmation"),
    EvidenceUpdate("supply:shipment", "unit_cost", 1, "demo: reviewed purchase unit-cost confirmation"),
]
ops.plan(previous=original.plan)  # Simulate other operator activity.
try:
    ops.apply_updates("supplier-expedite-002", confirmed_updates,
        expected_evidence_revision=comparison.evidence_revision,
        expected_operations_revision=comparison.operations_revision)
except StaleSnapshotError:
    print("Stale review token rejected; no confirmation applied.")
else:
    raise AssertionError("The old operations snapshot must not be accepted")
assert ops.store.current("supply:shipment", "available_day").value == 8
fresh_comparison = ops.compare_scenarios([candidate], previous=original.plan)
pprint([{k: row[k] for k in ("name", "total_score", "unfilled_units")} for row in fresh_comparison.rows()])


## 5. Apply confirmed fields once; reject a reused event ID with different content

The review action below deliberately submits the mock confirmation. Neither scenario ranking nor receipt replay silently approves an allocation. The illustrative fixed expedite fee remains an advisory estimate, not a booked payment.

In [ ]:
confirmed_receipt = ops.apply_updates("supplier-expedite-002", confirmed_updates,
    expected_evidence_revision=fresh_comparison.evidence_revision,
    expected_operations_revision=fresh_comparison.operations_revision)
state_after_confirmation = ops.revisions()
duplicate = ops.apply_updates("supplier-expedite-002", confirmed_updates,
    expected_evidence_revision=fresh_comparison.evidence_revision,
    expected_operations_revision=fresh_comparison.operations_revision)
assert duplicate.replayed
assert duplicate.evidence_ids == confirmed_receipt.evidence_ids
assert ops.revisions() == state_after_confirmation
pprint(duplicate.to_dict())
try:
    ops.apply_updates("supplier-expedite-002", [
        EvidenceUpdate("supply:shipment", "available_day", 4, "demo: changed payload")
    ], expected_evidence_revision=0, expected_operations_revision=0)
except IdempotencyConflict:
    print("Same event ID with different content rejected.")
else:
    raise AssertionError("An event identity cannot be reused for different facts")


## 6. Approve and commit a complete allocation group atomically

Planning creates proposals. Approval is explicit and checks the whole group's capacity. Local commit rechecks each decision and writes all new reservations and intent events in one transaction. If a later decision fails, earlier writes from that call roll back.

A committed plan means its allocation decisions were reserved locally; it does not mean a shipment occurred or that unmet demand vanished. Previously committed members are historical and are skipped on retries.

In [ ]:
current = ops.plan(previous=original.plan)
assert sum(current.plan.unfilled.values()) == 0
try:
    ops.commit_plan(current.plan_id)
except ValueError:
    print("Whole-plan commit correctly requires explicit approval.")
else:
    raise AssertionError("A proposed plan must not commit")
approved = ops.approve_plan(current.plan_id)
assert all(decision.status == "approved" for decision in approved)
committed = ops.commit_plan(current.plan_id)
pprint(committed)
assert not committed["idempotent"]
intents = [event for event in ops.events() if event["kind"] == "execution_intent"]
assert len(intents) == 2
history_after_commit = ops.events()
assert ops.commit_plan(current.plan_id)["idempotent"]
assert ops.events() == history_after_commit
remaining = ops.plan()
assert not remaining.plan.allocations
assert all(quantity == 0 for quantity in remaining.plan.unfilled.values())
print("Two local intents, ten units reserved, no duplicate intents on retry.")


## 7. Restart and replay an older message without restoring older facts

Close and reopen the same database. Deliver the original delay message again, using its original token. The stored receipt is returned, but the currently confirmed arrival remains day 3. A repeated plan commit likewise creates no new reservations or intents.

In [ ]:
ops.close()
restored = OperationsPipeline(db_path)
before_restart_replay = restored.revisions()
old_message_replay = restored.apply_updates("supplier-delay-001", delay_updates,
    expected_evidence_revision=delay_snapshot["evidence_revision"],
    expected_operations_revision=delay_snapshot["operations_revision"])
assert old_message_replay.replayed
assert old_message_replay.evidence_ids == delay_receipt.evidence_ids
assert restored.store.current("supply:shipment", "available_day").value == 3
assert restored.revisions() == before_restart_replay
assert restored.commit_plan(current.plan_id)["idempotent"]
assert len([e for e in restored.events() if e["kind"] == "execution_intent"]) == 2
assert sum(restored._reserved("order_id").values()) == 10
print("Persistent replay verified; older delay was not reapplied.")
print("External dispatch:", False)
restored.close()
workspace.cleanup()
